# 1. 파이썬의 메모리 관리 방식
- 1) Reference Counting (기본)
- 2) Generational Garbage Collection (보조, 순환 참조 해결용) - 별도의 GC가 처리
  
1) 기본은 Reference Counting  
2) 순환 참조는 Generational GC  
3) 대부분의 객체는 즉시 해제  
4) GC는 보조 장치  



## 1.1 Reference Counting (참조 카운트)

모든 객체는 ob_refcnt라는 참조 개수를 가집니다.

객체 생성 → refcount = 1  
변수에 할당 → +1  
다른 변수에 대입 → +1  
변수 삭제 → -1  
refcount == 0 → 즉시 메모리 해제  


In [8]:
import sys

a = []
print(sys.getrefcount(a))
b = a
print(sys.getrefcount(a))
del b
print(sys.getrefcount(a))

2
3
2


### 장점
- 즉시 메모리 해제 (deterministic)  
- 구현 단순
- 속도 빠름

### 단점
- 순환 참조를 해결 못함. 서로 참조하고 있기 때문에  
refcount가 0이 되지 않음 → 메모리 누수 발생

In [18]:
class A:
    pass

a = A()
b = A()
print(sys.getrefcount(a))  # 2
print(sys.getrefcount(b))  # 2

# 상호참조(a -> b, b -> a)
a.ref = b
b.ref = a
print(sys.getrefcount(a)) # 3
print(sys.getrefcount(b)) # 3

del a
print(sys.getrefcount(b)) # 3 <--- 참조가 줄어들지 않음. 2가 되어야 함.
del b



2
2
3
3
3


## 1.2 Generational GC (세대 기반 GC)

- 메모리를 3 영역으로 나눠서 관리
- 오래 살아남은 객체는 더 오래 살아남을 가능성이 높다.  
    
Generation 0  (young) <-- 새로 생성된 객체 저장(자주 검사)     
Generation 1          <-- 덜 자주 검사  
Generation 2  (old)   <-- GC후에도 살아남은 객체 저장(덜 검사)  

In [ ]:
import gc
print(gc.get_threshold())

(700, 10, 10)


- Gen0이 700번 증가하면 검사  
- Gen1은 Gen0이 10번 돌면 검사
- Gen2는 Gen1이 10번 돌면 검사

In [37]:
import gc
print(gc.get_count()) # (0세대의 객체수, 1세대의 객체수, 2세대의 객체수)

(558, 0, 0)


### 순환 참조 감지 방식

GC는 “추적 가능한 객체”만 검사. 모든 객체가 GC 대상은 아닙니다.

- GC 추적 대상
list  
dict  
set  
class instance  
user-defined object  

- GC 대상 아님  
int  
float  
str  
tuple (내부에 GC 객체 없으면)  

In [21]:
import gc
gc.is_tracked(123)  # False  리터럴은 GC대상 아님
gc.is_tracked([])   # True

True

### 누수 디버깅
- 디버그 플래그  
 
| 옵션            | 의미          |
| ------------- | ----------- |
| DEBUG_STATS   | 수집 통계 출력    |
| DEBUG_LEAK    | 수집 불가 객체 출력 |
| DEBUG_SAVEALL | 수집 안 하고 저장  |


In [41]:
gc.set_debug(gc.DEBUG_LEAK)

In [42]:
# 수집 불가 객체 확인
gc.garbage  # __del__이 있으면 순환 수집이 안될 수 있음 - Java의 finalize()

[]

### GC를 켜기/끄기
- 고성능 루프 돌릴 때
- 객체 엄청 많이 생성/삭제할 때
- 직접 메모리 관리 실험할 때

대량 객체 생성하는 배치 작업에서 GC 잠깐 끄면 성능 향상

In [ ]:
print(f"{gc.isenabled()=}")
gc.disable()
print(f"{gc.isenabled()=}")
gc.enable()
print(f"{gc.isenabled()=}")

gc.isenabled()=False
gc.isenabled()=False
gc.isenabled()=True


In [ ]:
gc.disable()  # GC 끄기
# 대량 객체 생성
gc.enable()   # GC 켜기
collected = gc.collect()  # 수동으로 GC 실행
print(collected)

177


In [ ]:
gc.get_objects()


[<function Pattern.match(string, pos=0, endpos=9223372036854775807)>,
 (<function Pattern.match(string, pos=0, endpos=9223372036854775807)>,
  Token.Name.Function.Magic,
  None),
 [(<function Pattern.match(string, pos=0, endpos=9223372036854775807)>,
   Token.Name.Variable.Magic,
   None)],
 <function Pattern.match(string, pos=0, endpos=9223372036854775807)>,
 (<function Pattern.match(string, pos=0, endpos=9223372036854775807)>,
  Token.Name.Variable.Magic,
  None),
 [(<function Pattern.match(string, pos=0, endpos=9223372036854775807)>,
   Token.Name.Decorator,
   None),
  (<function Pattern.match(string, pos=0, endpos=9223372036854775807)>,
   Token.Operator,
   None),
  (<function Pattern.match(string, pos=0, endpos=9223372036854775807)>,
   Token.Name,
   None)],
 <function Pattern.match(string, pos=0, endpos=9223372036854775807)>,
 (<function Pattern.match(string, pos=0, endpos=9223372036854775807)>,
  Token.Name.Decorator,
  None),
 <function Pattern.match(string, pos=0, endpos=92

### del 과 GC
예전 CPython에서는: 순환 참조 + del 있으면 GC가 수거 못함  
  
이유: 파괴 순서가 불명확하기 때문  

현재 CPython은 개선되었지만,  
여전히 __del__은 GC를 복잡하게 만듭니다.  

그래서 권장되는 방식:
- weakref
- context manager (with)
- 명시적 close()

In [61]:
class A:      # __del__이 있으면 순환 수집이 안될 수 있음.(Java의 finalize())
    def __del__(self):
        print("deleted")

### gc.garbage란?
Python은 참조 카운트 + 세대별 GC로 메모리를 관리  
순환 참조 객체는 참조 카운트가 0이 되지 않으므로 단순 reference counting으로는 해제되지 않는다.  
GC는 이런 순환 참조를 탐지해서 메모리를 해제하려 시도  
하지만 해제할 수 없는(uncollectable) 객체들이 있을 때, 이 객체들은 gc.garbage에 저장  

### 언제 gc.garbage가 채워지나?  
gc.garbage가 비어 있지 않게 되려면 다음이 필요합니다:  
1. GC 디버그 옵션 설정  

In [62]:
import gc
gc.set_debug(gc.DEBUG_SAVEALL)

2. GC 수동 실행  
이때 순환 참조가 있는 객체들 중 해제되지 않는 객체들이 gc.garbage로 남는다.

In [63]:
gc.collect()  # 수동으로 GC 실행

6748

### 왜 중요한가?
gc.garbage를 사용하는 대표적인 이유는 메모리 누수 디버깅:
- 프로그램에서 이상하게 메모리가 계속 증가할 때
- GC가 순환 참조를 해결 못 하고 있는 객체를 보고 싶을 때
- 이런 경우 gc.garbage를 확인하면 어떤 객체가 메모리에 남아있는지 확인 가능

| 항목           | 설명                                 |
| ------------ | ---------------------------------- |
| 모듈           | `gc` (Python 표준 가비지 컬렉터 모듈)        |
| `gc.garbage` | GC가 해제할 수 없는(uncollectable) 객체 리스트 |
| 기본 상황        | 대부분 빈 리스트                          |
| 사용 목적        | 메모리 누수 객체 확인, 디버깅                  |


In [66]:
import gc

gc.set_debug(gc.DEBUG_SAVEALL)
gc.collect()

for obj in gc.garbage:
    print(obj)


<function A.__del__ at 0x104f6bce0>
(<class 'object'>,)
{'__module__': '__main__', '__del__': <function A.__del__ at 0x104f6bce0>, '__dict__': <attribute '__dict__' of 'A' objects>, '__weakref__': <attribute '__weakref__' of 'A' objects>, '__doc__': None}
<class '__main__.A'>
(<class '__main__.A'>, <class 'object'>)
<attribute '__dict__' of 'A' objects>
<attribute '__weakref__' of 'A' objects>
(<class 'object'>,)
<class '__main__.A'>
(<class '__main__.A'>, <class 'object'>)
<attribute '__dict__' of 'A' objects>
{'__module__': '__main__', '__dict__': <attribute '__dict__' of 'A' objects>, '__weakref__': <attribute '__weakref__' of 'A' objects>, '__doc__': None}
<attribute '__weakref__' of 'A' objects>
[]
<zmq.Frame(b'aa66ab42-32a'...36B)>
[<zmq.Frame(b'aa66ab42-32a'...36B)>, <zmq.Frame(b'<IDS|MSG>')>, <zmq.Frame(b'4fcd8a88d7e9'...64B)>, <zmq.Frame(b'{"date":"202'...227B)>, <zmq.Frame(b'{}')>, <zmq.Frame(b'{"cellId":"v'...101B)>, <zmq.Frame(b'{"silent":fa'...184B)>]
<zmq.Frame(b'<IDS|MSG

AttributeError: '_SixMetaPathImporter' object has no attribute '_path'

### 약한 참조 (weakref)
- 순환 참조 해결용

In [ ]:
import weakref

class A:
    pass

a = A()
b = weakref.ref(a)  # weakref는 refcount를 증가시키지 않음


## PyPy - JIT 컴파일러를 내장
PyPy는 Python의 또 다른 구현체(인터프리터)  - RPython이라는 제한된 Python으로 작성  
보통 쓰는 건 CPython이고, PyPy는 그걸 대체할 수 있는 고성능 버전  
대규모 객체 생성 시스템이면 PyPy가 더 나을 수 있다.  
PyPy는 RPython이라는 제한된 Python으로 작성됐어.
- 반복문이 많은 알고리즘 코드
- 서버처럼 오래 실행되는 프로그램(워밍업이 필요)
- 재귀/루프 중심 처리

In [35]:
# 이처럼 객체를 많이 생성하는 코드는 CPython보다 Pypy가 유리
def loop():
    s = 0
    for i in range(10_000_000):
        s += i
    return s

loop()

49999995000000

### CPython과 PyPy와의 차이
- CPython은 단순하고 안정적인 구조
- PyPy는 공격적인 런타임 최적화 구조 

| 구분      | CPython | PyPy         |
| ------- | ------- | ------------ |
| 실행 방식   | 인터프리터   | 인터프리터 + JIT  |
| 속도      | 안정적     | 긴 실행에서 매우 빠름 |
| C 확장 호환 | 매우 좋음   | 일부 제약 있음     |
| GC | refcount 기반 | tracing GC      |
| 해재방식   | 즉시 해제       | 지연 해제           |
| 성능| predictable | high throughput |

Pypy가 적합한 경우  
-  알고리즘 문제 풀이
-  CPU bound 서버
-  반복 계산 많은 코드
-  순수 Python 로직이 대부분인 경우
  
Pypy가 적합하지 않은 경우  
- NumPy, Pandas heavy 프로젝트는 CPython만 지원
- 아주 짧게 실행되는 CLI 도구 <-- Pypy는 JIT컴파일러 워밍업이 필요

# 2. 성능 최적화

## 2.1 성능 측정
- 측정이 최우선 (Profiling)
- 최적화의 1원칙은: 느린 부분을 찾기 전에는 절대 최적화하지 마라

  
1. 측정하라 (profiling)  
2. 루프를 줄여라  
3. 자료구조를 바꿔라  
4. 알고리즘을 개선하라  
5. C 구현을 써라 (NumPy, Cython)  
6. GIL 이해하라

 ### 기본 도구
- timeit → 작은 코드 성능 비교
- cProfile → 전체 프로그램 병목 찾기
- line_profiler → 라인 단위 분석
- memory_profiler → 메모리 사용 분석

In [67]:
import cProfile

def work():
    sum(i*i for i in range(10_000_000))

cProfile.run("work()")

         10000444 function calls (10000438 primitive calls) in 1.328 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    1.118    1.118 4138730203.py:3(work)
 10000001    0.592    0.000    0.592    0.000 4138730203.py:4(<genexpr>)
        2    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
        4    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1384(_handle_fromlist)
      2/1    0.099    0.049    1.223    1.223 <string>:1(<module>)
        1    0.000    0.000    0.098    0.098 asyncio.py:206(_handle_events)
        1    0.000    0.000    0.000    0.000 asyncio.py:231(add_callback)
        4    0.000    0.000    0.000    0.000 attrsettr.py:43(__getattr__)
        4    0.000    0.000    0.000    0.000 attrsettr.py:66(_get_attr_opt)
        1    0.000    0.000    0.000    0.000 base_events.py:1859(_add_callback)
        1    0.000    0.000    0.006    0.006 

## 2.2 파이썬 레벨에서 할 수 있는 최적화
### 파이썬 루프 줄이기

In [70]:
%%time
result = []
for i in range(1000000):
    result.append(i * 2)


CPU times: user 59.5 ms, sys: 7.43 ms, total: 66.9 ms
Wall time: 70.5 ms


In [72]:
%%time
result = [i * 2 for i in range(1000000)]

CPU times: user 40.3 ms, sys: 7.62 ms, total: 47.9 ms
Wall time: 76.7 ms


In [73]:
%%time
result = list(map(lambda x: x*2, range(1000000)))

CPU times: user 60.2 ms, sys: 8.07 ms, total: 68.3 ms
Wall time: 92.8 ms


### 지역 변수 사용 - 전역 변수 접근은 느림

In [74]:
# 느림
def f():
    return math.sqrt(10)

# 빠름
def f():
    sqrt = math.sqrt
    return sqrt(10)


### 자료구조 선택이 성능을 좌우

| 작업              | 최적 자료구조             |
| --------------- | ------------------- |
| membership test | set                 |
| FIFO            | collections.deque   |
| 카운팅             | collections.Counter |
| 우선순위            | heapq               |


In [82]:
# 매우 느림
x = 5
if x in [range(1_000_000)]: # 하나씩 찾음. 느리다
    print(x)

# 빠름
if x in set(range(1_000_000)): # 빠름
    print(x)

5


In [ ]:
# 느림
s = ""
for i in range(10000):
    s += str(i) # 매번 새로운 객체를 생성

# 빠름
s = "".join(str(i) for i in range(10000))


### 병렬 처리 / 동시성
- GIL 이해하기 (중요)  
CPython은 GIL 때문에  
CPU-bound → 멀티스레드 효과 거의 없음  
I/O-bound → 스레드 효과 있음  

### NumPy 사용
- 파이썬 루프가 아니라 C 루프

In [89]:
%pip install numpy # numpy모듈 설치(jupytor안에 설치. 제일 확실)
import numpy as np   

a = np.arange(10_000_000)
a = a * 2



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Cython

In [90]:
cdef int i    # c로 컴파일

SyntaxError: invalid syntax (2910564099.py, line 1)

### PyPy 사용
- JIT 컴파일러
- 루프 많은 코드에서 빠름

### Numba 사용
- 파이썬 코드를 실행 직전에 컴파일해서 C처럼 빠르게 만들어주는 JIT 컴파일러  
- 파이썬은 기본적으로 인터프리터 언어 → 느림  
Numba는  
→ 코드를 실행하기 전에  
→ 기계어로 변환(JIT 컴파일)  
→ 그래서 속도향상  

In [95]:
%pip install numba
from numba import jit

@jit # jit가속 사용
def f():
    ...


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


ImportError: Numba needs NumPy 2.3 or less. Got NumPy 2.4.

### 메모리 최적화
1. slots   
- dict 제거
- 메모리 감소
- 속도 약간 증가

In [91]:
class Point:
    __slots__ = ('x', 'y')

2. generator사용

In [1]:
# 메모리 많이 씀
lst = [x*x for x in range(10_000_000)]

# 메모리 적음
gen = (x*x for x in range(10_000_000))
print(next(gen))
print(next(gen))
print(next(gen))

0
1
4
